# Lesson 03 — Building makemore Part 2: MLP

- **GitHub issue:** [#3](https://github.com/majorgilles/karpathy_ml_course/issues/3)
- **Video:** https://youtu.be/TCH_1BHY58I
- **Lesson guide:** [../README.md](../README.md)
- **Transcript:** [../transcript.md](../transcript.md)

Use this notebook for exploratory follow-along work. Move reusable code to `../src/`, lightweight checks to `../tests/`, and representative outputs to `../artifacts/`.

## 1. Load names and prepare the MLP workspace

This lesson moves beyond a bigram model: the MLP will use a fixed number of previous characters, called a **context window**, to predict the next character. First load the names as Python strings and establish the small set of libraries used for tensors, one-hot encoding, and later visualizations.

Each element of `words` is one name. The early cells inspect a few names and the dataset size before converting characters into numeric model inputs.


In [1]:
import torch  # Tensor operations and model parameters.
import torch.nn.functional as F  # One-hot encoding and other neural-network helpers.
import matplotlib.pyplot as plt  # Visualizations used later in the lesson.
from sympy.codegen.ast import float32  # Current exploration import; not used by these cells yet.
%matplotlib inline

In [2]:
# Load every name; each line in names.txt becomes one training sequence.
from pathlib import Path

candidate_paths = [
    Path("data/raw/names.txt"),  # Kernel launched from the repository root.
    Path("../../../data/raw/names.txt"),  # Kernel launched from this notebook folder.
]
names_path = next(path for path in candidate_paths if path.exists())
words = names_path.read_text(encoding="utf-8").splitlines()

words[:8]  # Inspect a small sample before building numeric examples.

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
# Confirm how many complete name sequences are available.
len(words)

32033

## 2. Map characters to model-friendly IDs

Neural networks work with numbers, not Python characters. `stoi` means string-to-index and maps each character to one stable integer; `itos` reverses that lookup for readable examples and generated output.

The boundary token `.` receives index `0`. It represents both left padding before a name and the end of a name, so the context window can start before any real letters have appeared.


In [4]:
# Collect the 26 lowercase letters once and sort them for reproducible IDs.
chars = sorted(list(set(''.join(words))))
# Reserve 0 for the boundary token, so letters begin at index 1.
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
# Invert the mapping: model indices back to printable characters.
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## 3. Turn names into fixed-context training examples

`block_size = 3` means every input row contains exactly three previous character IDs. `X` stores those three-character contexts and `Y` stores the one next-character target that followed each context.

For `emma`, the first example is `... → e`, then `..e → m`, and so on until `mma → .`. The initial three zeros are left padding with the boundary token. The dataset cell now iterates over all `words`, producing `N = len(X)` examples (`228,146` in the current run), so it avoids printing every transition.


In [ ]:
# Build one (context, next-character) example for every transition in every loaded name.
BLOCK_SIZE = 3  # Number of preceding characters the model receives as input.
X, Y = [], []  # X holds context rows; Y holds one next-character target per row.

for w in words:  # Iterate over the full training dataset.
    context = [0] * BLOCK_SIZE  # Start with three boundary-token IDs: "...".

    for ch in w + ".":  # Include the final boundary token as a target.
        ix = stoi[ch]  # Integer ID of the character this context should predict.
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]  # Drop the oldest ID and append this target ID.

# Convert Python lists to integer tensors for embedding lookup in the MLP.
X = torch.tensor(X)
Y = torch.tensor(Y)

In [6]:
# Verify: one three-ID context per example and one integer next-character target.
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

## 4. Look up embeddings for individual tokens, lists, and whole contexts

`C` is an embedding table with one row per vocabulary token. Each row contains two learnable features in this small example. Integer indexing retrieves rows from that table: one ID gives one vector, a list of IDs gives several vectors, and a tensor of context IDs gives an embedding vector for every token in every context.

The important input/output relationship is `C[X]`: `X` contains `N = len(X)` examples with 3 token IDs each, so the lookup replaces every ID with its two-feature embedding. The result has shape `(N, 3, 2)`: examples, context positions, embedding features. In the current full-dataset run, `N = 228,146`.


In [7]:
# Create a 27-row embedding table; each token initially receives two random features.
EMBEDDING_SIZE = 2
VOCAB_SIZE = 27
C = torch.randn((VOCAB_SIZE, EMBEDDING_SIZE))

In [8]:
# Direct lookup: one token ID selects one embedding row with two features.
C[5]

tensor([-1.6287,  0.6728])

In [9]:
# A list of IDs selects several rows, preserving the list order: output shape (3, 2).
C[[5, 6, 7]]

tensor([[-1.6287,  0.6728],
        [ 1.1374, -0.1316],
        [-0.8766,  0.7706]])

In [10]:
# Batched lookup: replace every ID in X with its C row, giving shape (N, 3, 2) where N = len(X).
emb = C[X]
emb

tensor([[[ 1.1761, -1.7041],
         [ 1.1761, -1.7041],
         [ 1.1761, -1.7041]],

        [[ 1.1761, -1.7041],
         [ 1.1761, -1.7041],
         [-1.6287,  0.6728]],

        [[ 1.1761, -1.7041],
         [-1.6287,  0.6728],
         [ 0.1345,  0.3292]],

        ...,

        [[ 0.6172, -1.5088],
         [ 0.6172, -1.5088],
         [-0.1855, -0.0506]],

        [[ 0.6172, -1.5088],
         [-0.1855, -0.0506],
         [ 0.6172, -1.5088]],

        [[-0.1855, -0.0506],
         [ 0.6172, -1.5088],
         [ 1.5766,  0.1454]]])

In [11]:
# Inspect the three axes: training examples, context positions, and embedding features.
emb.shape

torch.Size([228146, 3, 2])

## 5. Flatten each context into one MLP input row

`emb` has shape `(N, 3, 2)`, where `N = len(X)` is the number of training examples, 3 is the number of context positions, and 2 is the number of embedding features per position. The first linear layer expects one feature vector per example, so reshape combines the last two dimensions into six features while preserving the batch dimension.

`emb.reshape(-1, EMBEDDING_SIZE * BLOCK_SIZE)` therefore produces shape `(N, 6)`. The `-1` tells PyTorch to infer the number of examples. The `unbind` plus `cat` expression is an equivalent demonstration: split the three context positions into three `(N, 2)` tensors, then concatenate them into `(N, 6)`.

The comparison with `==` is elementwise, so it displays an `(N, 6)` grid of `True` values. `torch.equal` would instead return one Boolean answer for the entire tensors.


In [12]:
# First linear layer: six flattened context features feed 100 hidden neurons.
FIRST_HIDDEN_UNITS = 100
w1 = torch.randn((BLOCK_SIZE * EMBEDDING_SIZE, FIRST_HIDDEN_UNITS))
b1 = torch.randn(FIRST_HIDDEN_UNITS)  # One bias value per hidden neuron.

In [13]:
# Keep each example row and flatten its 3 × 2 context embedding into six MLP features.
reshaped = emb.reshape(-1, EMBEDDING_SIZE * BLOCK_SIZE)
# Elementwise check: every entry should be True because split-then-concatenate gives the same layout.
reshaped == torch.cat(torch.unbind(emb, dim=1), dim=1)

tensor([[True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        ...,
        [True, True, True, True, True, True],
        [True, True, True, True, True, True],
        [True, True, True, True, True, True]])

## 6. Produce hidden activations with a linear layer and `tanh`

Matrix multiplication combines each six-feature context row with `w1`: `(N, 6) @ (6, 100) → (N, 100)`, where `N = len(X)`. Adding `b1`, which has shape `(100,)`, broadcasts one bias per hidden neuron across all `N` examples.

$$
h = \tanh(\operatorname{flattened\_context} W_1 + b_1)
$$

Here, `h` is the `(N, 100)` hidden-activation tensor, `W_1` is `w1`, and `b_1` is `b1`. `tanh` applies to every raw pre-activation score and maps it into the range from `-1` to `1`. This nonlinearity matters because stacked linear layers alone could be reduced to one linear transformation; `tanh` lets the MLP represent more complex relationships between context characters and the next target.


In [14]:
# Inspect the input, weight, and matrix-product shapes of the first linear layer.
print(reshaped.shape)
print(w1.shape)
print((reshaped @ w1).shape)
# Compute 100 linear pre-activation scores per example, then apply tanh element by element.
h = torch.tanh(reshaped @ w1 + b1)
# Confirm: one 100-feature hidden activation vector for each of the N = len(X) examples.
h.shape

torch.Size([228146, 6])
torch.Size([6, 100])
torch.Size([228146, 100])


torch.Size([228146, 100])

## 7. Map hidden activations to one score per vocabulary candidate

Each of the `N = len(X)` hidden vectors has 100 features. The output layer maps those features to 27 raw scores, one for every vocabulary candidate: the 26 letters plus the boundary token `.`.

$$
\operatorname{logits} = h W_2 + b_2
$$

`W_2` is `w2` with shape `(100, 27)`, `b_2` is `b2` with shape `(27,)`, and `h` has shape `(N, 100)`. The resulting `logits` tensor has shape `(N, 27)`: one row per training example and one column per candidate target character. Logits are raw preference scores, not probabilities; the next cells convert each row into a probability distribution.


In [15]:
# Output-layer weights map 100 hidden features to one raw score for each vocabulary candidate.
w2 = torch.randn((FIRST_HIDDEN_UNITS, VOCAB_SIZE))
b2 = torch.randn(VOCAB_SIZE)  # One bias value per output vocabulary candidate.

In [16]:
# Compute one raw candidate score per vocabulary token for each training example: (N, 100) @ (100, 27) → (N, 27).
logits = h @ w2 + b2
logits.shape

torch.Size([228146, 27])

## 8. Convert logits into probability distributions with manual softmax

`logits` contains 27 raw candidate scores per training example. Exponentiation makes every score positive; these intermediate values are named `counts` by convention here, but they are not observed character counts. Dividing each row by its own total gives 27 probabilities that add to one.

$$
p_{i,j} = \frac{\exp(\ell_{i,j})}{\sum_{k=0}^{26} \exp(\ell_{i,k})}
$$

Here, `i` identifies one of the `N = len(X)` training examples, `j` identifies one candidate vocabulary token, `k` ranges over all 27 candidates, and `ℓ` is a logit. The expected target for example `i` is `Y[i]`; when the loss is introduced, `probs[i, Y[i]]` is the one probability that directly contributes to that example's negative log-likelihood.


In [17]:
# Exponentiate every raw score to create positive, unnormalized softmax values.
counts = logits.exp()

In [18]:
# Normalize each example's 27 candidate values by its row total; keepdim preserves shape (N, 1) for row-wise division.
probs = counts / counts.sum(1, keepdim=True)
# Verify the output shape and that two example distributions each use the full probability budget of 1.
print(probs.shape)
print(probs[0].sum())
print(probs[1].sum())

torch.Size([228146, 27])
tensor(1.0000)
tensor(1.0000)


## 9. Select the probability assigned to each expected target

`probs` contains 27 candidate probabilities for each of the `N = len(X)` training examples, while `Y` contains one expected target ID per example. Advanced indexing pairs row `i` with column `Y[i]`, selecting exactly the probability the model assigned to the observed next character.

For example 0, the selected value is `probs[0, Y[0]]`. `torch.arange(len(X))` supplies the row IDs `0` through `N - 1`, so `probs_for_labels` has shape `(N,)`: one expected-target probability per training example. This scores every example built from all `words`; it is full-dataset training loss, not held-out evaluation.


In [20]:
# Pair row i with its expected target ID Y[i] to select one target probability per current training example.
probs_for_labels = probs[torch.arange(len(X)), Y]
probs_for_labels.shape

torch.Size([228146])

## 10. Average the negative log-likelihood into one training loss

The loss should be small when the model gives a high probability to each expected target and large when it gives a low probability. Taking the logarithm makes probability values easier to combine, and the leading minus sign reverses the direction so better predictions have lower loss.

$$
\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log p_{i, Y_i}
$$

Here, `N = len(X)` is the number of training examples (`228,146` in the current full-dataset run), `p_{i, Y_i}` is `probs[i, Y[i]]`, and `Y_i` is the expected next-character ID. The `.mean()` computes the average loss across all `N` examples, not a total. This is full-dataset training loss, not a held-out evaluation score. Next, backpropagation calculates how each parameter can change to lower this average loss.


In [21]:
# Take negative logs of the expected-target probabilities, then average across all N = len(X) training examples.
loss = -probs_for_labels.log().mean()
loss

tensor(16.9216)

# Summary — from full data to mini-batch training

The full names dataset has produced `N = len(X)` supervised training examples (`228,146` in the current run): `X` contains three-token contexts and `Y` contains their expected next-token IDs. The next cells rebuild the MLP as one reproducible parameter set and repeatedly sample small groups of matching context-target rows for training.

The parameters begin random, then the training loop enables gradient tracking, calculates loss, and updates them. Each iteration uses a fresh random mini-batch rather than all `N` examples, making updates much faster while still learning from the full dataset over time.


In [22]:
# Confirm the full dataset has one three-token context and one target ID per training example.
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

## 11. Initialize one reproducible set of MLP parameters

A parameter is a number the model will later adjust to lower loss. The local random-number generator `g` starts from a fixed seed, so rerunning this cell with the same tensor shapes and order recreates the same random initial values. This makes experiments and explanations repeatable.

The model has five parameter tensors: embedding table `C`, first-layer weights `W1` and biases `b1`, then output-layer weights `W2` and biases `b2`. `parameters` groups them for counting now and for gradient-based updates later. `requires_grad=True` tells PyTorch to record how the loss depends on each tensor, so `loss.backward()` can later populate its gradient.


In [23]:
# Use a local fixed seed so this parameter initialization is reproducible.
g = torch.Generator().manual_seed(2147483647)
# Each parameter tracks gradients so backpropagation can measure its effect on loss.
C = torch.randn(VOCAB_SIZE, EMBEDDING_SIZE, requires_grad=True, generator=g)
W1 = torch.randn((BLOCK_SIZE * EMBEDDING_SIZE, FIRST_HIDDEN_UNITS), requires_grad=True, generator=g)
b1 = torch.randn(FIRST_HIDDEN_UNITS, requires_grad=True, generator=g)
W2 = torch.randn((FIRST_HIDDEN_UNITS, VOCAB_SIZE), requires_grad=True, generator=g)
b2 = torch.randn(VOCAB_SIZE, requires_grad=True, generator=g)
# Keep the five trainable tensors together for gradient clearing and updates.
parameters = [C, W1, b1, W2, b2]

In [24]:
# Count scalar values across all five parameter tensors; this configuration contains 3,481 numbers.
sum(p.nelement() for p in parameters)

3481

## 12. Train with random mini-batches

The full dataset has `N = len(X)` examples, but one update uses only `B = MINIBATCH_SIZE = 32` randomly selected rows. `torch.randint(0, X.shape[0], (B,))` creates `ix`, a length-`B` tensor of row IDs. It samples with replacement, so a row can appear more than once in a batch and not every example appears in a particular iteration.

The sampled tensors are `X[ix]` and `Y[ix]`, so their rows remain paired: each context still has its own expected next-character target. The mini-batch forward shape path is `(B, 3)` context IDs → `(B, 3, 2)` embeddings → `(B, 6)` flattened features → `(B, 100)` hidden activations → `(B, 27)` logits. `view(-1, EMBEDDING_SIZE * BLOCK_SIZE)` infers `B` from the selected rows.

`F.cross_entropy(logits, Y[ix])` combines softmax with average NLL for this mini-batch's `B` expected targets. Its printed value is a noisy training-loss estimate for the sampled rows, not full-dataset loss or held-out evaluation. Before backpropagation, setting each `p.grad` to `None` clears gradients accumulated from an earlier iteration. `loss.backward()` calculates gradients, and the final update subtracts `0.1 × gradient` from every parameter. This instructional `.data` update changes the model directly; an optimizer is the usual production interface. The loop runs 100 updates, drawing a fresh mini-batch each time.


In [42]:
# B is the number of training examples used for one fast, noisy gradient estimate.
MINIBATCH_SIZE = 32
print(X.shape)
print(Y.shape)

for _ in range(100):
    # Sample B row IDs uniformly with replacement from the full dataset.
    ix = torch.randint(0, X.shape[0], (MINIBATCH_SIZE,))
    # print(ix.shape) # torch.Size([32])

    # Forward pass only for the selected mini-batch: X[ix] and Y[ix] keep matching rows.
    # print(X[ix].shape) # (32, 3)
    emb = C[X[ix]]  # (B, 3, 2): look up every selected context-token embedding. #
    h = torch.tanh(emb.view(-1, EMBEDDING_SIZE * BLOCK_SIZE) @ W1 + b1)  # (B, 100) hidden activations.
    logits = h @ W2 + b2  # (B, 27) raw scores for every vocabulary candidate.
    # print(Y[ix].shape) # (32,)
    loss = F.cross_entropy(logits, Y[ix])  # Average NLL for this batch's B expected targets.

    # Backward pass: clear old accumulated gradients, then calculate new mini-batch loss gradients.
    for p in parameters:
        p.grad = None
    loss.backward()

    # Gradient-descent update: move each parameter opposite its gradient using learning rate 0.1.
    for p in parameters:
        p.data += -0.1 * p.grad

print(loss.item())  # Mini-batch training loss, not full-dataset or held-out loss.

torch.Size([228146, 3])
torch.Size([228146])
2.71960186958313
